# we will make a striped version of mamba

to do this we need to understand what the model is currently doing and see if we can change or update it at all

In [ ]:
#let's see what we have so far
# _name_: dna_embedding_caduceus
#then we can have some config parameters for the ssm
#then bidirectional add, and weight tie

#also let's see if we have mamba 2 and can make that an option
#indeed I have mamba 2, so we can make that an option too!

#it initializes a nn.model thing, then takes in a caduceus confdig as well as device, dtype and whether it's conjoined (I think the rcps) for train/test

#if it's conjoined, then complicated, but I'm not conjoining it here, that's only when I'm doing RCPS
#then just runs self.caduceus with return_dict false
#caduceus is defined by the caduceus config and device/dtype

In [ ]:
#looking at the modeling caduceus file, it is not too complicated.
#inherits the initialization and function form hugging faces PretrainedModel
#the way it initializes weights should work for transformers or SSMs? unclear

#then the actual model is built on caduceusmixermodel class
#this is the actual architecture, we can ignore a lot of it because we ignore things like the embeddings
#so if we alter this and ensure that it works with the initialization, then we are good?
#we can compare with the transformer class and see how it initializes and loads its backbone
#this is th elongconvlm script
#so combine ieas from the modeling caduceus and long conv lm to properly model it

#key ideas are we can ignore embeddings
#we might need to do the thing where we upsample and downsample from mamba to transformers
#but initialize it in a way that makes sense, and make sure the blocks work for either one

#could also use rope and base it on the striped hyena framework?
#I think better is jamba, since it does MOE too potentially? No need for ROPE, claim is mamba does that!
#it does sliding window attention and stuff too...

In [ ]:
#initializing, we can steal form jamba

# def _init_weights(self, module):
#         std = self.config.initializer_range
#         if isinstance(module, (nn.Linear, nn.Conv1d)):
#             module.weight.data.normal_(mean=0.0, std=std)
#             if module.bias is not None:
#                 module.bias.data.zero_()

In [ ]:
#we can also use Hydra instead of bimamba, so then we can do closer to jamba!
#make it adaptable so can do mamba2, mixed, or pure transformer
#let's us find a baesline to test it...
#no MOE for now, for now just make sure you get general idea of how it uses mamba and attention
#can implement MOE late if needed!

#bilby is quite complex tho lmao!
#basically, it requires a bunch of steps, but seems like the best approach for how to do the striped mamba! Let's use this to base it off of
# But understanding individual elements will be hard, so will have to manually run some elements myself to see the default what they are doing!
#But we use hydra and things like that instead of their implementations of mamba/transformer
#we should use ROPE for positional encoding despite what JAMBA says because of genomics and the pooling part

#we can test with no pooling and with pooling for transformer, but issue is memory limitations

#Still I think this is great!

In [ ]:
#so the first step is to get bilby working, and then to do it with hydra

#the nice part about hydra is that it uses mamba2 and no separate installation, so we can simply downbload the folder in and then import form there...
#but we can get the general structure and arguments and ideas about how to construct it from bilby
#but then we'll do our own thing with hydra and attention with ROPE!!



# updated details

## Key Features

- **Attention blocks**
  - Use RoPE + FlashAttention
  - Include gated attention  
    - Can copy implementation from the Qwen model
  - Double-check similarity with Enigma’s implementation
  - Verify RoPE behavior
    - Does RoPE get recalculated at each step?
    - Any issues during upsampling or downsampling?

- **Mamba2 / Hydra**
  - Implement Hydra core for Mamba2
  - make sure states aren't passed

- **U-Net structure**
  - Upsample in a U-Net–style fashion
    - Upsample to a specific target resolution
  - Ensure encoder/decoder symmetry
    - Match Enigma’s U-Net style

- **Modularity**
  - Design to support striped variants
  - Make upsampling factor configurable
  - Make sure it works with backbone



In [1]:
import math
from typing import Tuple

import torch
import torch.nn as nn
from einops import rearrange

try:
    from flash_attn.modules.mha import FlashSelfAttention
    from flash_attn.ops.triton.rotary import apply_rotary
except ImportError:
    FlashSelfAttention = None
    apply_rotary = None

# from jaxtyping import Bool, Float, Integer
class MockJaxType:
    def __class_getitem__(cls, item):
        # This swallows the parameters like [Tensor, "batch seq"]
        # and just returns the class itself, effectively ignoring them.
        return cls

# Assign the dummy class to the names you need
Bool = MockJaxType
Float = MockJaxType
Integer = MockJaxType
#we don't have jaxtyping, so let's just mock it
from torch import Tensor

In [2]:
FlashSelfAttention, apply_rotary #ok we have both!!

(flash_attn.modules.mha.FlashSelfAttention,
 <function flash_attn.ops.triton.rotary.apply_rotary(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, seqlen_offsets: Union[int, torch.Tensor] = 0, cu_seqlens: Optional[torch.Tensor] = None, max_seqlen: Optional[int] = None, interleaved=False, inplace=False, conjugate=False) -> torch.Tensor>)

In [8]:
#so we went through the Qwen3 attention part and it's pretty obvious and straight forward! But we can easilyy see how it can translate to enigma

#we looked at enigma code, pretty straightforward, but does require some changes to the MHA

class MHAGate(nn.Module):
    """Multi-head gated self-attention using FlashAttention.

    This is adapted from
    https://github.com/Dao-AILab/flash-attention/blob/934f6ad714691a21a09b78c3e19a2378917e9cba/flash_attn/modules/mha.py#L373
    to support unpadded variable length inputs with RoPE positional encoding.
    
    Also adds an optional gating mechanism on the output of the attention, adapted from
    https://github.com/qiuzh20/gated_attention/blob/main/modeling_qwen3.py#L236-L290
    """

    def __init__(
        self,
        dim: int,
        head_dim: int,
        use_flash_attn: bool = True,
        use_rope: bool = True,
        use_gating: bool = True,
        rope_base: float = 20000.0,
        rope_interleaved: bool = False,
        use_alibi: bool = False,
        window_size: Tuple[int, int] = (-1, -1),
        dropout: float = 0.0,
        use_qk_norm: bool = False,
        bias: bool = False,
        device: str | torch.device | None = None,
    ):
        super().__init__()

        if use_flash_attn: #we do want to use flash attention
            if FlashSelfAttention is None or apply_rotary is None:
                raise ImportError(
                    "`flash_attn` must be installed when `use_flash_attn=True`"
                )
        else:
            if use_alibi: #we can also do alibi or window size stuff since we have flash attention
                raise ValueError(
                    "`use_alibi=True` is only supported when `use_flash_attn=True`"
                )
            if window_size != (-1, -1):
                raise ValueError(
                    "`window_size` is only supported when `use_flash_attn=True`"
                )
        self.use_flash_attn = use_flash_attn

        if dim % head_dim != 0:
            raise ValueError(
                f"dim must be divisible by head_dim, got dim={dim} and "
                f"head_dim={head_dim}"
            )

        self.num_heads = dim // head_dim

        self.use_rope = use_rope
        if self.use_rope:
            rope_dim = head_dim  # use head_dim for RoPE
            self.rotary_emb = RotaryEmbedding(
                dim=rope_dim, #dim should be head dim for rope since it's the dim of q and k. It's the size per head cuz applies per head
                base=rope_base, #this is the frequency base which follows base^(-2i/dim). For us we want it to be really large since we have long context length
                interleaved=rope_interleaved, #false just makes it more efficient in some ways. Shouldn't matter that much
                device=device,
            )
            #RoPE should be calculated once and then we're good since we'll always use the same l ength
            #and it indeed does cache the sin and cos, so we're good

        self.use_alibi = use_alibi
        if self.use_alibi:
            alibi_slopes = torch.tensor(get_alibi_slopes(self.num_heads), device=device)
        else:
            alibi_slopes = None

        if self.use_flash_attn:
            self.self_attn = FlashSelfAttention(
                attention_dropout=dropout,
                window_size=window_size,
                alibi_slopes=alibi_slopes,
            )
        else:
            self.self_attn = SelfAttention(attention_dropout=dropout)

        self.use_gating = use_gating
        if self.use_gating:
            self.multiplier = 4
        else:
            self.multiplier = 3
        qkv_dim = head_dim * self.num_heads * self.multiplier
        # qkvg_dim = head_dim * self.num_heads * 4  # extra for gating
        self.Wqkv = nn.Linear(dim, qkv_dim, bias=bias)

        self.use_qk_norm = use_qk_norm
        if self.use_qk_norm:
            raise NotImplementedError("RMSNorm is not available in this PyTorch version. Use Qwen3's implementation.")
            self.q_norm = nn.RMSNorm(head_dim)
            self.k_norm = nn.RMSNorm(head_dim)

        self.out_proj = nn.Linear(dim, dim, bias=bias)

    def forward(
        self,
        x: Float[Tensor, "b l d"] | Float[Tensor, "total_tokens d"],
        key_padding_mask: Bool[Tensor, "b l"] | None = None,
        cu_seqlens: Integer[Tensor, " b+1"] | None = None,
        max_seqlen: int | None = None,
    ):
        """
        If `cu_seqlens` and `max_seqlen` are provided, the input is assumed to
        be unpadded (i.e. shape of (total_tokens, d)).

        Args:
            x: (batch, seqlen, hidden_dim) or (total_tokens, hidden_dim) if unpadded
            key_padding_mask: boolean mask, True means to keep, False means to mask out.
                (batch, seqlen). Only applicable when not using FlashAttention.
            cu_seqlens: Tensor of shape (batch_size + 1,) containing the cumulative
                sequence lengths of the sequences in the batch, used to index into qkv.
            max_seqlen: Maximum sequence length in the batch.
        """
        # First check if the input is valid
        if cu_seqlens is not None:
            assert max_seqlen is not None
            assert key_padding_mask is None
            assert self.use_flash_attn
        if key_padding_mask is not None:
            assert cu_seqlens is None
            assert max_seqlen is None
            assert not self.use_flash_attn

        # Projection to qkv
        #we will modify it here since we need to do gating, we have to do a separate 
        
        qkv = self.Wqkv(x)
        qkv = rearrange(
            qkv, "... (numcomp h d) -> ... numcomp h d", numcomp=self.multiplier, h=self.num_heads
        )
        if self.use_gating:
            # q,k,v,gate = qkv.unbind(dim=-3)  # shape (..., h, d)
            # qkv = torch.stack([q, k, v], dim=-3)  # shape (..., 3, h, d)
            gate = qkv[..., 3, :, :]  # shape (..., h, d)
            qkv = qkv[..., :3, :, :]  # shape (..., 3, h, d)

        # gate = torch.sigmoid(gate)

        if self.use_rope:
            qkv = self.rotary_emb(qkv, cu_seqlens=cu_seqlens, max_seqlen=max_seqlen)

        if self.use_qk_norm:
            qkv = self._apply_qk_norm(qkv)

        # To support both with and without FlashAttention, create a kwargs for self_attn
        kwargs = (
            {"cu_seqlens": cu_seqlens, "max_seqlen": max_seqlen}
            if self.use_flash_attn
            else {"key_padding_mask": key_padding_mask}
        )
        output = self.self_attn(qkv, **kwargs)

        # Apply gating
        if self.use_gating:
            gate = torch.sigmoid(gate)
            output = output * gate
        

        # Projection to output
        out = rearrange(output, "... h d -> ... (h d)") #concat heads
        
        out = self.out_proj(out)

        return out

    def _apply_qk_norm(
        self, qkv: Float[Tensor, "... three h d"]
    ) -> Float[Tensor, "... three h d"]:
        """Apply QK normalization to stacked `qkv`

        FlashAttention can be more efficient if it is already stacked [1]
        (although I have not tested whether this is true after the overhead of this
        operation). Therefore, rather than splitting `qkv` into `q`, `k`, and `v`,
        apply the normalization to the stacked `qkv` and return the stacked `qkv`
        again.

        [1] https://github.com/Dao-AILab/flash-attention/blob/04adaf0e9028d4bec7073f69e4dfa3f6d3357189/flash_attn/flash_attn_interface.py#L1236-L1238  # noqa E501
        """
        q, k, v = qkv.unbind(dim=-3)  # return views so shouldn't be much overhead
        q = self.q_norm(q)
        k = self.k_norm(k)

        return torch.stack(
            [q, k, v], dim=-3
        )  # stack's overhead should be minimal compared to the other operations
        
a = MHAGate(dim=4096, head_dim=128, use_flash_attn=True, use_rope=False, use_gating=True, device='cuda:0')
a.to('cuda:0')

MHAGate(
  (self_attn): FlashSelfAttention(
    (drop): Dropout(p=0.0, inplace=False)
  )
  (Wqkv): Linear(in_features=4096, out_features=16384, bias=False)
  (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
)

In [9]:
#let's test the things step byy step
#first as input

test_input = torch.randn(4,1024,4096) #batch x seq_len x dim
test_input = test_input.to('cuda:0')
qkv = a.Wqkv(test_input)


In [10]:
qkv.shape #it's collapsed all together?

torch.Size([4, 1024, 16384])

In [11]:
qkv = rearrange(
    qkv, "... (numcomp h d) -> ... numcomp h d", numcomp=a.multiplier, h=a.num_heads
)
qkv.shape
#ok very logical, splits it to batch x seq_len x qkvg x heads x head_dim

torch.Size([4, 1024, 4, 32, 128])

In [12]:
a.use_gating

True

In [13]:
gate = qkv[..., 3, :, :]  # shape (..., h, d)
qkv = qkv[..., :3, :, :]  # shape (..., 3, h, d)
gate.shape, qkv.shape

(torch.Size([4, 1024, 32, 128]), torch.Size([4, 1024, 3, 32, 128]))

In [ ]:
# qkv = a._apply_qk_norm(qkv) oh we can't because RMS norm isn't in pytorch for this version lol. Can implement it ourself or use Qwen version?

AttributeError: 'MHAGate' object has no attribute 'q_norm'

In [14]:
kwargs = (
    {"cu_seqlens": None, "max_seqlen": None}
    if a.use_flash_attn
    else {"key_padding_mask": None}
)
kwargs

{'cu_seqlens': None, 'max_seqlen': None}

In [ ]:
#cast qkv to float 16
qkv = qkv.to(torch.float16)
output = a.self_attn(qkv, **kwargs)
output.shape
#oh it did the whole thing the multiply by v too

torch.Size([4, 1024, 32, 128])

In [16]:
gate = torch.sigmoid(gate)
output = output * gate
output.shape

torch.Size([4, 1024, 32, 128])

In [17]:
out = rearrange(output, "... h d -> ... (h d)")
out.shape

torch.Size([4, 1024, 4096])

In [ ]:
out = a.out_proj(out)
out.shape
#yup this works eaclty as we expect!!

torch.Size([4, 1024, 4096])

In [ ]:
#now the main thing is we need to get rope working, let's see
#so first we need cu_seqlens. This is primarily for variable position lengths. We can see if it's by default 

#ok wait it doesn't need to be provided. If it isn't provided we see that it auto assumes the shape. The part where it iwould be put into rotary embeddings is
#shape [4, 1024, 3, 32, 128]
#it gets the max shape
#but does it auto calculate cu_seqlens? let's see
#if we provide cu_seqlens then it assumes we collapsed it down. But since we have fixxed length, it's not needed it's fine!

# test MHA

In [22]:
import sys
sys.path.append('/data1/lesliec/sarthak/caduceus/src/')
from models.sequence.striped_mamba import MHAGate

a = MHAGate(dim=4096, head_dim=128, use_flash_attn=True, use_rope=True, use_gating=True, device='cuda:0')
a.to('cuda:0')
#make it float 16
a = a.half()

test_input = torch.randn(4,1024,4096) #batch x seq_len x dim
test_input = test_input.to('cuda:0')
#and make it float 16
test_input = test_input.half()

test_output = a(test_input)

In [23]:
test_output.shape

torch.Size([4, 1024, 4096])

In [25]:
a

MHAGate(
  (rotary_emb): RotaryEmbedding()
  (self_attn): FlashSelfAttention(
    (drop): Dropout(p=0.0, inplace=False)
  )
  (Wqkv): Linear(in_features=4096, out_features=16384, bias=False)
  (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
)

In [ ]:
#hey that's what we expected lol. Same shape?? Ok so we did it I think, implemented MHA properly! Last step is to do the RMS norm

In [2]:
#one more time but let's use the QWEN3 RMS norm
import sys
sys.path.append('/data1/lesliec/sarthak/caduceus/src/')
from models.sequence.striped_mamba import MHAGate
import torch

a = MHAGate(dim=4096, head_dim=128, use_flash_attn=True, use_rope=True, use_gating=True, device='cuda:0', use_qk_norm=True)
a.to('cuda:0')
#make it float 16
a = a.half()

test_input = torch.randn(4,1024,4096) #batch x seq_len x dim
test_input = test_input.to('cuda:0')
#and make it float 16
test_input = test_input.half()

test_output = a(test_input)

In [3]:
test_output.shape

torch.Size([4, 1024, 4096])

In [ ]:
a #see the normalizataion is as expected!

MHAGate(
  (rotary_emb): RotaryEmbedding()
  (self_attn): FlashSelfAttention(
    (drop): Dropout(p=0.0, inplace=False)
  )
  (Wqkv): Linear(in_features=4096, out_features=16384, bias=False)
  (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
  (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
)

In [1]:
import sys
sys.path.append('/data1/lesliec/sarthak/caduceus/src/')
from models.sequence.striped_mamba import MHAGate